# Проект спринта 27. Часть 2: эксперименты и обучение

Это вторая тетрадка проекта по извлечению именованных сущностей (NER).
Чтобы приступить к работе с ней, вы должны завершить работу с первой тетрадкой,`ydp_27p_eda.ipynb`: ииследовали данные и изучили логику токенизации и выравнивания меток.

В этой части проекта вы настроите оценку качества моделей в целом и на уровне целых сущностей, получите baseline, проведёте два эксперимента с разным объёмом обучающих данных, сравните модели, проанализируете ошибки и проверите итоговую модель на собственных текстах.

**Модель:** `bert-base-multilingual-cased`.

### Где выполнять эту часть проекта

Задания второй части требуют GPU, поэтому выполняйте её на виртуальной машине. Подключение к ней описано в уроке «Инструкция к виртуальной машине с GPU».

Поминте, что время работы VM ограничено 10 часами и начинает расходоваться с момента её инициализации. Запускайте машину, когда будете готовы обучать модели, и не забывайте отключать её, когда прекращаете работу.

Брать заготовки из первой тетрадки не нужно: код в разделе «Подготовка данных» далает всё необходимое

### Как работать с тетрадкой

Выполняйте ячейки последовательно. Дополняйте ячейки с пометкой `TODO`, сохраняйте
результаты вычислений и записывайте выводы в соответствующих ячейках Markdown.
Тестовую выборку не используйте для выбора настроек.

Структура тетрадки:

- Подготовка окружения и обеспечение воспроизводимости.
- Подготовка данных.
- Шаг 4 — подготовка к обучению и создание базовой модели.
- Шаг 5 — эксперимент на малой выборке.
- Шаг 6 — эксперимент на увеличенной выборке.
- Шаг 7 — сравнение результатов.
- Шаг 8 — анализ ошибок.
- Шаг 9 — сохранение модели и inference.
- Шаг 10 — выводы и самопроверка по чек-листу.

## Подготовка окружения и обеспечение воспроизводимости

Установите зависимости, импортируйте библиотеки, зафиксируйте `SEED` и проверьте
доступное вычислительное устройство.

In [ ]:
# Установка библиотек
%pip install -q datasets transformers accelerate evaluate seqeval spacy


In [ ]:
from collections import Counter
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
from datasets import load_dataset
from seqeval.metrics.sequence_labeling import get_entities
from spacy import displacy
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
    set_seed,
)


In [ ]:
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")
print("Версии: " + ", ".join(
    f"{package}={version(package)}"
    for package in [
        "torch",
        "transformers",
        "datasets",
        "accelerate",
        "evaluate",
        "seqeval",
        "spacy",
    ]
))


## Подготовка данных для BERT

Этот блок повторяет подготовку из первой тетрадки: загружает WikiANN, собирает словари
меток, токенизирует предложения и выравнивает NER-метки с подтокенами BERT. Код приведён целиком — дополнять его не нужно.

Напомним политику выравнивания: первый токен слова получает исходную метку,
остальные токены, специальные токены и паддинг получают `-100` и исключаются
из расчёта функции потерь.

In [ ]:
# Загрузка русскоязычной конфигурации WikiANN и словарей меток
raw_datasets = load_dataset("unimelb-nlp/wikiann", "ru")

label_list = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
ENTITY_TYPES = sorted({
    label.split("-")[-1]
    for label in label_list
    if label != "O"
})

print(f"Метки: {label_list}")
print(f"Типы сущностей: {ENTITY_TYPES}")

In [ ]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [ ]:
def tokenize_and_align_labels(examples):
    """Токенизирует слова и выравнивает NER-метки с подтокенами."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# Эти переменные понадобятся для работы моделей
train_full = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]
test_dataset = tokenized_datasets["test"]

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Контрольная проверка: длины полей одного примера должны совпадать
example = train_full[0]
assert (
    len(example["input_ids"])
    == len(example["attention_mask"])
    == len(example["labels"])
)

print(f"Обучающая выборка:   {len(train_full)}")
print(f"Валидационная:       {len(eval_dataset)}")
print(f"Тестовая:            {len(test_dataset)}")

## Подготовка к обучению

Ниже предсавлены несколько блоков с кодом, которые будут нужны вам для работы с моделями:

- код, который фиксирует общие параметры экспериментов;
- функция для инициализации свежей модели;
— функция для обучения модели.

Работая с моделями, используйте валидационный срез для выбора `learning_rate`, а полную тестовую выборку — только для итоговой оценки моделей. Между экспериментами меняется размер обучающей выборки, остальные настройки остаются одинаковыми.


In [ ]:
# Настройки моделей
FEWSHOT_SIZE = 100
LARGE_SIZE = 1000
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EVAL_SUBSET_SIZE = 1000
CANDIDATE_LEARNING_RATES = (5e-5, 2e-5)

eval_small = eval_dataset.shuffle(seed=SEED).select(range(EVAL_SUBSET_SIZE))
print(
    f"Валидация во время обучения: "
    f"{len(eval_small)} из {len(eval_dataset)} примеров"
)


In [ ]:
def init_model():
    """Создаёт свежую модель token classification с одинаковой инициализацией."""
    set_seed(SEED)
    return AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )


In [ ]:
def train_model(train_dataset, lr, output_dir):
    """Создаёт свежую модель и обучает её с заданным learning rate."""
    model = init_model()
    steps_per_epoch = int(np.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        # Сохраняем чекпойнт каждой эпохи и в конце берём лучший по валидационному F1
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=lr,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=max(steps_per_epoch // 2, 1),
        seed=SEED,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_small,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer


all_results = {}

## Настройка метрик

Реализуйте преобразование логитов и меток в строковые IOB2-теги. Исключите позиции
`-100`. `compute_metrics` должен возвращать общие `precision`, `recall`, `F1`,
`accuracy` и `precision`, `recall`, `F1` по каждому типу сущности. Используйте
ключ `f1` для общего F1 и ключи вида `PER_f1` для отдельных типов меток. Если тип
отсутствует в результате `seqeval`, верните для него `0.0`.


In [ ]:
metric = evaluate.load("seqeval")


def align_predictions(predictions, labels):
    """Преобразует логиты и id-метки в строки, исключая позиции -100."""
    # TODO: выберите класс с максимальным логитом
    # TODO: удалите позиции -100 и преобразуйте id в label_list
    raise NotImplementedError


def compute_metrics(p):
    """Возвращает общие метрики и метрики по PER, ORG, LOC."""
    predictions, labels = p
    # TODO: вызовите align_predictions и metric.compute
    # TODO: соберите словарь общих метрик и метрик по ENTITY_TYPES
    raise NotImplementedError


In [ ]:
def evaluate_model(trainer, dataset, name):
    """Выполняет один predict и возвращает метрики и строковые теги."""
    output = trainer.predict(dataset)
    true_predictions, true_labels = align_predictions(
        output.predictions,
        output.label_ids,
    )
    results = metric.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    print(f"\n=== {name} ===")
    print(
        f"{'Overall':8} | "
        f"P: {results['overall_precision']:.4f} | "
        f"R: {results['overall_recall']:.4f} | "
        f"F1: {results['overall_f1']:.4f}"
    )
    for entity in ENTITY_TYPES:
        if entity in results:
            print(
                f"{entity:8} | "
                f"P: {results[entity]['precision']:.4f} | "
                f"R: {results[entity]['recall']:.4f} | "
                f"F1: {results[entity]['f1']:.4f}"
            )
        else:
            print(f"{entity:8} | не предсказан ни разу")

    return results, true_predictions, true_labels


## Шаг 4. Создание базовой модели

Baseline — предобученный BERT с новой необученной NER-головой. Не вызывайте
`train()`: сразу оцените свежую модель на полной тестовой выборке.


In [ ]:
baseline_model = init_model()
baseline_trainer = Trainer(
    model=baseline_model,
    args=TrainingArguments(
        "bert-ner-baseline",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
    ),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

baseline_results, _, _ = evaluate_model(
    baseline_trainer,
    test_dataset,
    "Baseline без обучения",
)
all_results["baseline"] = baseline_results

del baseline_trainer, baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Выводы о базовой модели

> **TODO:** Запишите свои выводы после прогона baseline-модели.


> 💾 **Точка сохранения.** Первые метрики получены. Закоммитьте тетрадку, не откладывая дело в долгий ящик:
>
> ```bash
> git add . && git commit -m "Baseline без дообучения" && git push
> ```

## Шаг 5. Эксперимент на малой выборке

Выберите 100 воспроизводимых примеров из полной обучающей выборки. Обучите две свежие
модели с `learning_rate=5e-5` и `learning_rate=2e-5`. Выберите лучший вариант по
`F1` на одном и том же валидационном срезе.


In [ ]:
# TODO: перемешайте train_full с SEED и выберите FEWSHOT_SIZE примеров
# train_100 = ...


In [ ]:
# TODO: обучите вариант A с первым learning_rate
# lr_a = CANDIDATE_LEARNING_RATES[0]
# trainer_lr_a = ...
# val_f1_a = ...

# TODO: выведите F1 варианта A на валидационном срезе


In [ ]:
# TODO: обучите независимый вариант B со вторым learning_rate
# lr_b = CANDIDATE_LEARNING_RATES[1]
# trainer_lr_b = ...
# val_f1_b = ...

# TODO: выведите F1 варианта B на валидационном срезе


In [ ]:
# TODO: выберите лучший learning_rate только по валидационному F1
# best_lr = ...
# trainer_fewshot = ...

# TODO: оцените выбранную модель на полном test_dataset
# Сохраните fewshot_results, pred_tags_fewshot и true_tags_test
# Добавьте fewshot_results в all_results

# Освободите модели, которые больше не нужны


### Выводы о модели, дообученной на малой выборке

> **TODO:** укажите валидационный `F1` для двух значений `learning_rate`, обоснуйте выбор лучшей модели и определите, какой тип сущностей распознаётся хуже всего на тесте.


> 💾 **Точка сохранения.** Позади два обучения — самое время сохранить результат
> в репозиторий, чтобы не повторять их заново.

## Шаг 6. Эксперимент на увеличенной выборке

Выберите 1000 примеров из полной обучающей выборки, не используйте данные, на которых обучалась первая модель.

Инициализируйте новую модель, используйте лучший `learning_rate`, не меняйте остальные
настройки.


In [ ]:
# TODO: сформируйте train_1000 из train_full с тем же SEED

# TODO: обучите свежую модель с best_lr и сохраните полученный объект

# TODO: оцените её на полном test_dataset
# Сохраните large_results, pred_tags_large и true_tags_large
# Проверьте, что true_tags_large совпадают с true_tags_test
# Добавьте large_results в all_results


> 💾 **Точка сохранения.** Это был самый долгий прогон модели в проекте! Закоммитьте тетрадку с посчитанными метриками прямо сейчас.

## Шаг 7. Сравнение результатов

Соберите в одной таблице данные по baseline и моделям, обученным на 100 и 1000 примерах.

Покажите общие `precision`, `recall`, `F1` и `F1` для `PER`, `ORG`, `LOC`.


In [ ]:
def results_to_row(results):
    """Преобразует результаты seqeval в строку сравнительной таблицы."""
    # TODO: добавьте overall precision, recall, F1 и F1 по ENTITY_TYPES
    raise NotImplementedError


# TODO: сформируйте comparison_df из all_results и выведите таблицу


In [ ]:
# TODO: рассчитайте абсолютный и относительный прирост метрик
# при переходе от 100 к 1000 примерам

# TODO: определите тип сущности с наибольшим приростом F1


### Выводы о результатах сравнения моделей

> **TODO:** сравните baseline и две обученные модели. Укажите абсолютный и относительный прирост общего `F1`, изменение `precision` и `recall`, а также тип сущности с наибольшим приростом качества предсказания.


> 💾 **Точка сохранения.** Сравнительная таблица — ключевой результат проекта.
> Отправьте её в репозиторий.

## Шаг 8. Анализ ошибок

Проанализируйте ошибки на уровне целых сущностей. Используйте сохранённые предсказания, чтобы не запускать повторный `predict()` на полной тестовой выборке.

При анализе ориентируйтесь на перечисленные в таблице типы ошибок:

| Тип | Описание |
|---|---|
|Пропуск сущности|Сущность есть в эталоне, но отсутствует в предсказании.|
|Неверный тип|Границы совпали, но тип сущности отличается.|
|Неправильные границы|Предсказание пересекается с эталоном, но границы отличаются.|
|Лишняя сущность|Модель нашла сущность там, где её нет в эталоне.|


In [ ]:
test_tokens = raw_datasets["test"]["tokens"]

MISSED = "пропуск сущности"
WRONG_TYPE = "неверный тип"
WRONG_BOUNDARIES = "неправильные границы"
SPURIOUS = "лишняя сущность"
ERROR_KINDS = [MISSED, WRONG_TYPE, WRONG_BOUNDARIES, SPURIOUS]


def classify_errors(true_tags, pred_tags):
    """Сопоставляет сущности и классифицирует расхождения."""
    unmatched_pred = list(get_entities(pred_tags))
    errors = []

    for gold in get_entities(true_tags):
        gold_type, gold_start, gold_end = gold

        exact = next(
            (
                pred
                for pred in unmatched_pred
                if (pred[1], pred[2]) == (gold_start, gold_end)
            ),
            None,
        )
        if exact is not None:
            unmatched_pred.remove(exact)
            if exact[0] != gold_type:
                errors.append({
                    "kind": WRONG_TYPE,
                    "gold": gold,
                    "pred": exact,
                })
            continue

        overlapping = next(
            (
                pred
                for pred in unmatched_pred
                if pred[1] <= gold_end and gold_start <= pred[2]
            ),
            None,
        )
        if overlapping is not None:
            unmatched_pred.remove(overlapping)
            errors.append({
                "kind": WRONG_BOUNDARIES,
                "gold": gold,
                "pred": overlapping,
            })
            continue

        errors.append({"kind": MISSED, "gold": gold, "pred": None})

    for spurious in unmatched_pred:
        errors.append({
            "kind": SPURIOUS,
            "gold": None,
            "pred": spurious,
        })

    return errors


def format_entity(entity, tokens):
    """Возвращает сущность в виде «текст» [ТИП]."""
    if entity is None:
        return "—"
    entity_type, start, end = entity
    return f"«{' '.join(tokens[start:end + 1])}» [{entity_type}]"


### Поиск примеров ошибок

Выведите не менее пяти предложений, в которых ошибается модель, обученная на 1000 примеров. Для каждого покажите текст, эталонные и предсказанные теги, тип ошибки и сущности.


In [ ]:
NUM_ERROR_EXAMPLES = 5

# TODO: пройдите по true_tags_test и pred_tags_large
# TODO: выведите первые пять примеров, где classify_errors вернул ошибки


### Сравнение частот ошибок

Посчитайте типы ошибок по всей тестовой выборке для обеих обученных моделей.
Сравните абсолютные значения и доли, затем определите самый частый тип ошибки.


In [ ]:
def count_error_kinds(true_tags_list, pred_tags_list):
    """Считает частоты типов ошибок по всему набору."""
    counter = Counter()
    for true_tags, pred_tags in zip(true_tags_list, pred_tags_list):
        for error in classify_errors(true_tags, pred_tags):
            counter[error["kind"]] += 1
    return counter


# TODO: постройте таблицу частот и долей ошибок для двух моделей
# TODO: выведите самый частый тип ошибки модели на 1000 примерах


### Поиск расхождений между моделями

Найдите примеры, в которых каждая из обученных моделей показывает себя лучше другой. Для этого продемонстрируйте случаи, где модель на малой выборке права, а модель на увеличенной ошибается, и наоборот. Сравнивайте извлечённые сущности, а не только последовательности тегов.


In [ ]:
def same_entities(tags_a, tags_b):
    return get_entities(tags_a) == get_entities(tags_b)


def find_disagreements(true_tags_list, tags_a, tags_b, max_examples=2):
    """Возвращает индексы примеров, где одна модель права, а другая ошибается."""
    a_better = []
    b_better = []

    for i, true_tags in enumerate(true_tags_list):
        a_ok = same_entities(tags_a[i], true_tags)
        b_ok = same_entities(tags_b[i], true_tags)

        if a_ok and not b_ok and len(a_better) < max_examples:
            a_better.append(i)
        if b_ok and not a_ok and len(b_better) < max_examples:
            b_better.append(i)
        if len(a_better) >= max_examples and len(b_better) >= max_examples:
            break

    return a_better, b_better


# TODO: вызовите find_disagreements для двух обученных моделей
# TODO: выведите текст, эталон и оба предсказания для найденных примеров


### Выводы из анализа ошибок

> **TODO:** укажите самый частый тип ошибки, сравните общее число ошибок двух моделей, приведите примеры расхождений и сформулируйте ограничения модели.
> Учитывайте возможный шум автоматической разметки WikiANN.


> 💾 **Точка сохранения.** Анализ ошибок готов — время сделать коммит.

## Шаг 9. Сохранение модели и проверка на новых текстах

Используйте итоговую модель из эксперимента на увеличенной выборке. Получите
предсказания на небольшой фиксированной части тестовой выборки, сохраните модель
и токенизатор, загрузите их обратно и повторите предсказание на тех же примерах.
Результаты до и после загрузки должны полностью совпасть.


In [ ]:
# TODO: сохраните и повторно загрузите итоговую модель.
#
# 1. Получите предсказания на небольшой части test_dataset.
# 2. Сохраните модель через save_model(...) или save_pretrained(...),
# 3. Загрузите модель через AutoModelForTokenClassification.from_pretrained(...).
# 4. Повторите предсказание и проверьте совпадение через np.array_equal(...).


### Проверка модели на собственных текстах

Создайте 3–5 предложений, которых нет в WikiANN. Желательно включить однословные
и многословные сущности `PER`, `ORG`, `LOC`, а также сложные или неоднозначные примеры.


In [ ]:
# TODO: инициализируйте ner_pipeline через pipeline(...).
# Передайте модель, загруженную на предыдущем шаге, tokenizer,
# aggregation_strategy="first" и доступное устройство.


def visualize_predictions(text, pipeline_output):
    """Визуализирует предсказанные сущности через displacy."""
    ents = [
        {
            "start": entity["start"],
            "end": entity["end"],
            "label": entity["entity_group"],
        }
        for entity in pipeline_output
    ]
    displacy.render(
        {"text": text, "ents": ents, "title": None},
        style="ent",
        manual=True,
        jupyter=True,
    )


In [ ]:
custom_sentences = [
    # TODO: добавьте 3–5 собственных предложений
]

# TODO: Прогоните их через pipeline и визуализируйте через visualize_predictions

> 💾 **Точка сохранения.** Модель сохранена и проверена на своих текстах.
> Закоммитьте тетрадку: веса в репозиторий не попадут, они исключены в `.gitignore`.

## Шаг 9. Финальные выводы

> **TODO:** сформулируйте итог проекта своими словами. Проанализируйте:
>
> - выбранный `learning_rate` и основание его выбора;
> - отличие baseline от дообученных моделей;
> - как увеличение обучающей выборки повлияло на качество;
> - изменение качества по `PER`, `ORG`, `LOC`;
> - основные типы ошибок и ограничения;
> - результат проверки сохранения и инфененс.

Отдельный блок посвятите рекомендациями заказчику. Объясните менеджерам «Эхолота», какие объёмы данных для разметки планировать и на какие детали обращать внимание при обучении. Отдельно проговорите нюансы, связанные с тем, что вы экспериментировали с энциклопедичными данными, а не с теми, которые обычно обрабатывает сервис медиамониторинга.



## Ресурсы

- [Hugging Face: Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)
- [seqeval](https://github.com/chakki-works/seqeval)
- [WikiANN](https://huggingface.co/datasets/unimelb-nlp/wikiann)
- [bert-base-multilingual-cased](https://huggingface.co/bert-base-multilingual-cased)


## Чек-лист готовности второй части проекта

### Эксперименты

- [ ] Настройки экспериментов собраны в одном месте, `SEED` зафиксирован.
- [ ] Метрики считаются через `seqeval` на уровне сущностей, позиции `-100` исключены.
- [ ] Получен baseline без вызова `train()`.
- [ ] Проверены два `learning_rate`, лучший выбран по валидационному `F1`.
- [ ] Обе обученные модели оценены на одной полной тестовой выборке.
- [ ] Все три варианта модели представлены в одной сравнительной таблице.
- [ ] Рассчитан абсолютный и относительный прирост качества после увеличения выборки.

### Анализ и итог

- [ ] Показано минимум пять ошибок модели на увеличенной выборке.
- [ ] Посчитаны частоты типов ошибок.
- [ ] Найдены примеры, в которых каждая из двух обученных моделей превосходит другую.
- [ ] Лучшая модель сохранена и повторно загружена, предсказания совпадают.
- [ ] Выполнен инференс на 3–5 собственных предложениях с демонстрацией результатов.
- [ ] Написаны выводы по каждому блоку и финальный вывод, в котором зафиксированы конкретные значения и ограничения.
- [ ] Предоставлены рекомендации компании «Эхолот».
- [ ] Описаны нюансы, связанные с отличиями текстов Википедии и «Эхолот».

### Формальные требования

- [ ] Во всех случайных операциях используется единый `SEED`.
- [ ] Все обязательные `TODO` заполнены, временные заглушки удалены.
- [ ] Ядро перезапущено, тетрадка выполнена сверху вниз без ошибок.
- [ ] Результаты вычислений, таблицы и визуализации сохранены в тетрадке.
- [ ] Веса моделей не добавлены в репозиторий.